# End-to-End Project — Hands-On Practice

This notebook converts the project scripts (train_model.py,
export_and_optimize.py, deploy_api.py) into interactive exercises.

## Exercise 1: Train the SmallCnn on CIFAR-10

Train for a few epochs and save the checkpoint.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)


class SmallCnn(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


# Quick training (2 epochs for demo)
torch.manual_seed(0)
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_ds = datasets.CIFAR10(root="./data", train=True, download=True, transform=tfm)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)

model = SmallCnn()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

for epoch in range(1, 3):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        opt.zero_grad(set_to_none=True)
        loss = crit(model(x), y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * x.size(0)
    print(f"Epoch {epoch}: avg_loss={total_loss/len(train_ds):.4f}")

print(f"Training complete. Model has {sum(p.numel() for p in model.parameters()):,} params.")

## Exercise 2: Export to ONNX and Validate

In [ ]:
import os
import numpy as np
import onnx
from onnx import checker

model.eval()
onnx_path = "/tmp/cifar_model.onnx"

dummy = torch.randn(1, 3, 32, 32)
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    do_constant_folding=True,
)

# Validate
m = onnx.load(onnx_path)
checker.check_model(m)
print(f"ONNX export validated: {onnx_path}")
print(f"File size: {os.path.getsize(onnx_path) / 1024:.1f} KB")

# Parity check
import onnxruntime as ort

x_test = torch.randn(4, 3, 32, 32)
with torch.no_grad():
    y_pt = model(x_test).numpy()

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
y_ort = sess.run(None, {"input": x_test.numpy()})[0]

print(f"Max abs diff: {np.max(np.abs(y_pt - y_ort)):.2e}")
print(f"Parity: {'PASS' if np.allclose(y_pt, y_ort, atol=1e-5) else 'FAIL'}")

## Exercise 3: Dynamic INT8 Quantization

In [ ]:
import os
from onnxruntime.quantization import quantize_dynamic, QuantType

quantized_path = "/tmp/cifar_model_int8.onnx"
quantize_dynamic(
    model_input=onnx_path,
    model_output=quantized_path,
    weight_type=QuantType.QUInt8,
)

orig_size = os.path.getsize(onnx_path)
quant_size = os.path.getsize(quantized_path)
print(f"Original: {orig_size/1024:.1f} KB")
print(f"Quantized: {quant_size/1024:.1f} KB")
print(f"Size reduction: {(1 - quant_size/orig_size)*100:.1f}%")

# Compare accuracy
sess_q = ort.InferenceSession(quantized_path, providers=["CPUExecutionProvider"])
y_q = sess_q.run(None, {"input": x_test.numpy()})[0]

# Check argmax stability
argmax_orig = np.argmax(y_ort, axis=1)
argmax_quant = np.argmax(y_q, axis=1)
print(f"Argmax match: {np.all(argmax_orig == argmax_quant)}")
print(f"FP32 vs INT8 max diff: {np.max(np.abs(y_ort - y_q)):.4f}")

## Exercise 4: Inference API Handler (deploy_api.py pattern)

In [ ]:
import numpy as np
import onnxruntime as ort

CIFAR_MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32).reshape(1, 3, 1, 1)
CIFAR_STD = np.array([0.2023, 0.1994, 0.2010], dtype=np.float32).reshape(1, 3, 1, 1)
CIFAR_LABELS = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]


def preprocess_image(img_array: np.ndarray) -> np.ndarray:
    """Preprocess RGB image array to CIFAR-normalized tensor."""
    from PIL import Image
    img = Image.fromarray(img_array).convert("RGB").resize((32, 32), Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0
    chw = np.transpose(arr, (2, 0, 1))[None, :, :, :]
    return (chw - CIFAR_MEAN) / CIFAR_STD


def predict(sess: ort.InferenceSession, x: np.ndarray) -> dict:
    """Run prediction and return top-5 results."""
    inp_name = sess.get_inputs()[0].name
    out_name = sess.get_outputs()[0].name
    logits = sess.run([out_name], {inp_name: x.astype(np.float32)})[0].reshape(-1)

    # Softmax
    ex = float(np.max(logits))
    probs = np.exp(logits - ex)
    probs = probs / np.sum(probs)

    top5 = np.argsort(-probs)[:5]
    return {
        "top_label": CIFAR_LABELS[int(np.argmax(probs))],
        "confidence": float(probs[np.argmax(probs)]),
        "top5": [
            {"label": CIFAR_LABELS[int(i)], "prob": float(probs[int(i)])}
            for i in top5
        ],
    }


# Demo with random image
fake_img = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
x = preprocess_image(fake_img)

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
result = predict(sess, x)
print(f"Predicted: {result['top_label']} ({result['confidence']:.2%})")
print("Top-5:")
for r in result["top5"]:
    print(f"  {r['label']}: {r['prob']:.4f}")

## Summary

In this notebook you practiced the complete end-to-end pipeline:
1. Training a CNN on CIFAR-10
2. Exporting to ONNX with validation and parity testing
3. Applying dynamic INT8 quantization and measuring size/accuracy trade-offs
4. Building the inference handler matching the deploy_api.py pattern